## Feature Engineering y pipeline de preprocesamiento

### Proyecto:
Pacientes con problemas de hígado - India

**Autor:** Mariana Bedoya Arismendy

**Fecha:** 2026-08-19

**Incidencia:** 10 - Feature Engineering

### Descripción:

En este notebook preparamos los datos de pacientes con posibles problemas hepáticos
para el entrenamiento de un modelo de clasificación que prediga el diagnóstico a partir
de un panel de pruebas de función hepática. El punto de partida son los hallazgos del
análisis exploratorio: valores faltantes repartidos en todas las variables, 60 filas
duplicadas, 3 registros con bilirrubinas fisiológicamente imposibles, pruebas hepáticas
con colas derechas muy marcadas y redundancias entre variables clínicamente relacionadas.

A partir de esos hallazgos construimos, con `Pipeline` y `ColumnTransformer` de
scikit-learn, un preprocesador que se encarga de la imputación, la transformación
logarítmica de las pruebas sesgadas, la creación de dos variables clínicas derivadas,
el escalado y la codificación de `Gender`. El preprocesador se ajusta únicamente con el
conjunto de entrenamiento, de modo que puede transformar datos nuevos sin repetir
manualmente ningún paso y sin fuga de información entre entrenamiento y prueba.

## 📚 Importar librerías

Importamos las librerías base del proyecto: `pandas` y `numpy` para manipular los datos,
y los componentes de `scikit-learn` que usaremos para construir los pipelines
(`Pipeline`, `ColumnTransformer`, imputadores, codificadores y escaladores). La clase
`CocientesClinicos`, definida más adelante, hereda de `BaseEstimator` y
`TransformerMixin` para respetar la interfaz de transformadores de scikit-learn.

In [1]:
# librerías base para ciencia de datos
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

## 💾 Cargar datos

Cargamos el dataset desde el archivo Parquet de la etapa intermedia, que contiene las
11 variables clínicas con los tipos de datos ya corregidos durante la exploración.
Verificamos que el archivo exista antes de leerlo y construimos la ruta de forma
relativa a la raíz del proyecto, de modo que el notebook funcione en cualquier
instalación.

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
DATA_PATH = DATA_DIR / "02_intermediate/Pacientes_porblemas_higado_india_type_fixed.parquet"

assert DATA_PATH.exists(), f"No se encuentra el archivo del proyecto: {DATA_PATH}"

pacientes_df = pd.read_parquet(DATA_PATH, engine="pyarrow")

print(f"Dataset cargado: {pacientes_df.shape[0]} filas y {pacientes_df.shape[1]} columnas")
pacientes_df.head()

Dataset cargado: 663 filas y 11 columnas


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.9,1
1,62,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.0,1
4,72,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.4,1


In [3]:
# versiones de las librerías para reproducibilidad
print("Pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("sklearn version:", sk.__version__)

Pandas version: 2.3.3
numpy version: 2.3.5
sklearn version: 1.9.0


## 📊 Revisión de la calidad de los datos

Antes de definir cualquier transformación confirmamos el estado real del dataset:
estructura, valores faltantes, duplicados, comportamiento de la variable objetivo y
consistencia entre variables. Los notebooks de exploración y análisis ya caracterizaron
estos puntos; aquí los verificamos de nuevo sobre el archivo que vamos a procesar,
porque las decisiones de limpieza deben tomarse sobre lo que los datos son hoy y no
sobre lo que se recuerda del análisis.

El objetivo de esta revisión no es repetir el EDA completo, sino identificar exactamente
qué problemas hay que resolver antes de construir los pipelines.

In [4]:
pacientes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 663 entries, 0 to 662
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Age                         662 non-null    Int64   
 1   Gender                      655 non-null    category
 2   Total_Bilirubin             657 non-null    Float64 
 3   Direct_Bilirubin            656 non-null    Float64 
 4   Alkaline_Phosphotase        646 non-null    Float64 
 5   Alamine_Aminotransferase    644 non-null    Float64 
 6   Aspartate_Aminotransferase  651 non-null    Float64 
 7   Total_Protiens              656 non-null    Float64 
 8   Albumin                     661 non-null    Float64 
 9   Albumin_and_Globulin_Ratio  659 non-null    Float64 
 10  Dataset                     648 non-null    category
dtypes: Float64(8), Int64(1), category(2)
memory usage: 54.1 KB


La estructura coincide con lo documentado en la exploración: 663 registros y 11
variables. La edad (`Age`) es un entero que admite nulos, las ocho pruebas de función
hepática son decimales, y `Gender` y `Dataset` son categóricas. No hay columnas de texto
libre, fechas ni identificadores de paciente.

`Dataset` es la variable objetivo: sus valores son las etiquetas de texto `'1'`
(paciente con problemas de hígado) y `'2'` (paciente sin problemas).

In [5]:
nulos_por_columna = pacientes_df.isna().sum().sort_values(ascending=False)
print(f"Celdas con valores faltantes: {nulos_por_columna.sum()} de {pacientes_df.size}")
nulos_por_columna

Celdas con valores faltantes: 98 de 7293


Alamine_Aminotransferase      19
Alkaline_Phosphotase          17
Dataset                       15
Aspartate_Aminotransferase    12
Gender                         8
Direct_Bilirubin               7
Total_Protiens                 7
Total_Bilirubin                6
Albumin_and_Globulin_Ratio     4
Albumin                        2
Age                            1
dtype: int64

Hay 98 celdas con valores faltantes, repartidas en las 11 variables y en proporciones
pequeñas (la más afectada es `Alamine_Aminotransferase`, con 19 nulos). Esto tiene dos
consecuencias para la etapa actual:

1. El objetivo `Dataset` tiene 15 valores faltantes: esos registros no tienen etiqueta
   y no sirven para entrenar un modelo supervisado. Se descartan en la limpieza; la
   variable objetivo no se imputa.
2. Los faltantes de las variables predictoras se imputan dentro del pipeline de
   scikit-learn, con parámetros aprendidos únicamente del conjunto de entrenamiento,
   para que la imputación sea reproducible en datos nuevos y no genere fuga de
   información.

In [6]:
duplicados_exactos = int(pacientes_df.duplicated().sum())
filas_en_grupos = int(pacientes_df.duplicated(keep=False).sum())
print(
    f"Filas duplicadas exactas: {duplicados_exactos} ({duplicados_exactos / len(pacientes_df):.2%})"
)
print(f"Filas que pertenecen a grupos de duplicados: {filas_en_grupos}")
print("\nDistribución de la variable objetivo (1: problemas de hígado, 2: sin problemas):")
print(pacientes_df["Dataset"].value_counts(dropna=False))

Filas duplicadas exactas: 60 (9.05%)
Filas que pertenecen a grupos de duplicados: 111

Distribución de la variable objetivo (1: problemas de hígado, 2: sin problemas):
Dataset
1      460
2      188
NaN     15
Name: count, dtype: int64


Hay 60 filas duplicadas exactas (el 9,05 % del dataset) y, en total, 111 filas
pertenecen a alguno de los 51 grupos de repetidos. Como el dataset no tiene identificador
de paciente, no es posible saber si se trata del mismo individuo registrado varias veces
o de pacientes distintos con resultados idénticos, algo posible en un panel de pruebas
con valores mayoritariamente discretos.

Entre los pacientes con etiqueta, la clase `'1'` (problemas de hígado) tiene 460 casos y
la clase `'2'` (sin problemas) 188: un desbalance moderado de aproximadamente 71 % / 29 %
que conviene conservar en la división entrenamiento/prueba mediante estratificación.

In [7]:
# la bilirrubina directa es una fracción de la total: nunca debería superarla
inconsistente = pacientes_df["Direct_Bilirubin"] > pacientes_df["Total_Bilirubin"]
pacientes_df.loc[inconsistente, ["Total_Bilirubin", "Direct_Bilirubin", "Dataset"]]

,Total_Bilirubin,Direct_Bilirubin,Dataset
246,1.8,9.0,1
261,1.5,7.0,1
279,1.0,1.4,1


Las tres filas mostradas violan una restricción fisiológica: la bilirrubina directa es
una fracción de la bilirrubina total, así que nunca puede superarla. En dos de los casos
la violación es flagrante (total 1,8 con directa 9,0; total 1,5 con directa 7,0) y
tienen toda la apariencia de errores de transcripción. Sin acceso a la fuente original
no es posible corregirlas, por lo que se descartan en la limpieza.

En resumen, la revisión deja tres problemas estructurales por resolver antes de separar
los datos (duplicados, filas sin etiqueta y bilirrubinas inconsistentes) y dos problemas
que se resuelven dentro del pipeline (valores faltantes de las predictoras y fuertes
asimetrías de las pruebas hepáticas). Los valores extremos de las pruebas clínicas no se
eliminan: el análisis exploratorio concluyó que son clínicamente posibles en pacientes
graves, y más adelante se explica cómo se gestionan.

## 🧹 Limpieza previa a la separación de los datos

Las tres operaciones de limpieza se hacen **antes** de dividir los datos, y el orden
importa:

1. **Duplicados exactos.** Si un registro idéntico quedara a la vez en entrenamiento y
   en prueba, el modelo sería evaluado con datos que ya vio: una forma clara de fuga de
   información. Por eso la deduplicación no puede hacerse después del split. Tras
   eliminarlos, el dataset pasa de 663 a 603 filas.
2. **Filas sin etiqueta.** Los 15 registros con `Dataset` faltante no pueden usarse en
   aprendizaje supervisado. La variable objetivo no se imputa (inventar diagnósticos no
   tiene sentido clínico), así que las filas se descartan.
3. **Bilirrubinas inconsistentes.** Las 3 filas con bilirrubina directa mayor que la
   total son errores de captura sin posibilidad de corrección y se descartan.

Sobre los **valores atípicos**: no se elimina ninguna observación por ser estadísticamente
extrema. El análisis exploratorio mostró que los máximos de las pruebas hepáticas (AST
4929, ALT 2000, fosfatasa alcalina 2110, bilirrubina total 75) son clínicamente posibles
en pacientes con daño hepático severo; eliminarlos sería descartar justamente a los
pacientes más enfermos, que son los que el modelo necesita aprender a identificar. El
efecto de sus colas largas se gestiona de forma reproducible con la transformación
logarítmica del pipeline, que comprime los valores altos sin perderlos.

In [8]:
# 1. Duplicados exactos: se eliminan antes del split para que registros idénticos
#    no terminen simultáneamente en entrenamiento y prueba.
filas_iniciales = len(pacientes_df)
limpio_df = pacientes_df.drop_duplicates()
print(f"Tras eliminar duplicados: {len(limpio_df)} filas (-{filas_iniciales - len(limpio_df)})")

# 2. Registros sin etiqueta en el objetivo: no sirven para el entrenamiento supervisado.
filas_sin_etiqueta = int(limpio_df["Dataset"].isna().sum())
limpio_df = limpio_df[limpio_df["Dataset"].notna()]
print(f"Tras descartar filas sin etiqueta: {len(limpio_df)} filas (-{filas_sin_etiqueta})")

# 3. Filas con bilirrubina directa mayor que la total: errores de captura
#    fisiológicamente imposibles y sin fuente para corregirlos.
#    Las comparaciones con valores faltantes dan NA: se marcan como no
#    inconsistentes para no descartar filas por tener pruebas incompletas.
inconsistente = (limpio_df["Direct_Bilirubin"] > limpio_df["Total_Bilirubin"]).fillna(False)
limpio_df = limpio_df[~inconsistente]
print(f"Tras descartar filas inconsistentes: {len(limpio_df)} filas (-{int(inconsistente.sum())})")

print("\nDistribución de clases en los datos limpios:")
print(limpio_df["Dataset"].value_counts())

Tras eliminar duplicados: 603 filas (-60)
Tras descartar filas sin etiqueta: 588 filas (-15)
Tras descartar filas inconsistentes: 585 filas (-3)

Distribución de clases en los datos limpios:
Dataset
1    414
2    171
Name: count, dtype: int64


La limpieza deja el dataset en 585 filas: 663 → 603 al quitar duplicados → 588 al quitar
filas sin etiqueta → 585 al quitar las bilirrubinas inconsistentes. El desbalance de
clases se mantiene casi intacto, con 414 pacientes con problemas de hígado (70,8 %) y
171 sin problemas (29,2 %), así que las decisiones de limpieza no alteraron la
composición del problema.

In [9]:
nulos_restantes = limpio_df.drop(columns=["Dataset"]).isna().sum()
filas_con_nulos = limpio_df.drop(columns=["Dataset"]).isna().any(axis=1).sum()
print(f"Valores faltantes restantes en las predictoras: {nulos_restantes.sum()} celdas")
print(f"Filas afectadas: {filas_con_nulos} de {len(limpio_df)}")
nulos_restantes.sort_values(ascending=False)

Valores faltantes restantes en las predictoras: 54 celdas
Filas afectadas: 25 de 585


Alamine_Aminotransferase      12
Alkaline_Phosphotase           9
Aspartate_Aminotransferase     8
Total_Protiens                 7
Albumin_and_Globulin_Ratio     4
Gender                         4
Direct_Bilirubin               4
Total_Bilirubin                3
Albumin                        2
Age                            1
dtype: int64

Tras la limpieza quedan 54 valores faltantes, todos en variables predictoras y
concentrados en pocas filas (25 de 585). Ninguna variable tiene un volumen de nulos que
justifique descartarla, así que se imputan dentro del pipeline: con la mediana en las
numéricas (robusta frente a las colas de las pruebas hepáticas) y con la moda en
`Gender`, que conserva 4 faltantes y cuya categoría mayoritaria (`Male`) representa
cerca del 76 % de los valores informados.

### Por qué la mediana en las variables numéricas

La elección de la mediana no es una preferencia arbitraria: viene directamente de la
forma que tienen estas variables en nuestro dataset, documentada en los notebooks de
exploración y análisis.

**Mediana frente a media.** Las cinco pruebas hepáticas tienen asimetrías entre 3,2 y
10,6, así que su media queda arrastrada por los valores extremos de los pacientes más
graves. El caso más claro es AST, con media 111 frente a mediana 43 según el EDA, más
del doble; algo parecido ocurre con la bilirrubina total (media 3,38 frente a mediana
1,0) y la fosfatasa alcalina (media 285 frente a mediana 208). Imputar con la media
asignaría a un paciente con la prueba faltante un valor que en realidad corresponde a
un caso bastante enfermo; la mediana, en cambio, es el valor central robusto: representa
al paciente típico sin dejarse arrastrar por las colas. En las variables simétricas
(`Age`, `Total_Protiens`, `Albumin`, `Albumin_and_Globulin_Ratio`) la media y la mediana
prácticamente coinciden (45,0 y 45 en la edad, por ejemplo), de modo que usar la mediana
también allí no cuesta nada y mantiene una sola estrategia coherente para todas las
numéricas.

**Mediana frente a cero.** El cero no es un valor real observable en ninguna de estas
variables: los mínimos del dataset son 4 años para la edad, 0,4 para la bilirrubina
total, 63 para la fosfatasa alcalina, 10 para las transaminasas y 0,9 para la albúmina.
Rellenar un faltante con cero inventaría un valor clínicamente imposible (un paciente
sin bilirrubina ni enzimas en sangre) y desplazaría artificialmente la distribución hacia
abajo, justo en las variables que el EDA identificó como las que mejor separan a los
enfermos de los sanos.

**Mediana frente a la eliminación de filas.** Solo 25 de las 585 filas (el 4,3 %)
tienen algún faltante en las predictoras y ninguna variable supera el 2,1 % de nulos.
Cada una de esas filas conserva la etiqueta del diagnóstico y la mayor parte de sus
mediciones. En un dataset ya pequeño tras la limpieza, y con una clase minoritaria de
apenas 171 pacientes, descartarlas sería regalar observaciones etiquetadas que el modelo
necesita. La eliminación de registros se reservó para los casos en que estaba
justificada: filas sin etiqueta y filas con datos imposibles de corregir.

**Mediana frente a la moda.** La moda solo tiene sentido para variables categóricas.
Estas variables son continuas y toman entre 40 y 261 valores distintos, así que el valor
más frecuente no representa ningún centro de la distribución; por eso la moda se reserva
para `Gender`.

**Mediana frente a métodos de imputación más complejos.** Alternativas como
`KNNImputer` o `IterativeImputer` pueden estimar los faltantes a partir de otras
variables, pero aquí el beneficio esperado es marginal: los nulos son pocos (54 celdas
en total), las pruebas clínicas están en escalas muy distintas y con colas muy marcadas,
y el dataset tiene solo 585 registros, poco para que esos métodos ganen estabilidad.
Preferimos una imputación simple, robusta y transparente; evaluar alternativas más
sofisticadas y comparar su impacto en el desempeño queda anotado en la sección de
propuestas.

En todos los casos la mediana se aprende dentro del pipeline con el conjunto de
entrenamiento y se aplica igual a la prueba y a datos nuevos, de modo que la imputación
es reproducible y no depende de cálculos manuales sobre el DataFrame.

## ✂️ Separación de variables y división entrenamiento / prueba

Separamos las 10 variables predictoras del objetivo y convertimos `Dataset` a una
variable binaria: 1 para problemas de hígado y 0 para sin problemas. Esta conversión se
hace fuera del pipeline de características y de forma explícita, porque la variable
objetivo no debe entrar al preprocesador: no se imputa, no se escala ni se codifica
junto con las predictoras. Con valores 0/1 queda lista para los clasificadores de
scikit-learn.

La división es estratificada por el objetivo para conservar el desbalance 71/29 en ambos
conjuntos, y se fija `random_state=42` para que los resultados sean reproducibles.

In [10]:
# variables predictoras: todas las columnas excepto el objetivo
X_features = limpio_df.drop(columns=["Dataset"])

# objetivo binario: 1 = problemas de hígado ('1'), 0 = sin problemas ('2')
y_target = (limpio_df["Dataset"] == "1").astype(int)

# 80 % entrenamiento, 20 % prueba, estratificado por el objetivo
x_train, x_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, stratify=y_target, random_state=42
)

print(f"Entrenamiento: {x_train.shape[0]} filas ({int(y_train.sum())} con problemas de hígado)")
print(f"Prueba: {x_test.shape[0]} filas ({int(y_test.sum())} con problemas de hígado)")
print(f"Proporción de la clase 1 en entrenamiento: {y_train.mean():.1%}")
print(f"Proporción de la clase 1 en prueba: {y_test.mean():.1%}")

Entrenamiento: 468 filas (331 con problemas de hígado)
Prueba: 117 filas (83 con problemas de hígado)
Proporción de la clase 1 en entrenamiento: 70.7%
Proporción de la clase 1 en prueba: 70.9%


El conjunto de entrenamiento queda con 468 pacientes (331 con problemas de hígado,
70,7 %) y el de prueba con 117 (83, 70,9 %): el desbalance original se conserva en
ambos, que es justo lo que busca la estratificación.

### ¿Por qué dividir antes de transformar?

Todas las transformaciones que aprenden parámetros a partir de los datos (la mediana de
imputación, la media y la desviación del escalado, las categorías del one-hot) deben
ajustarse **únicamente con el conjunto de entrenamiento**. Si se calcularan con todos los
datos, información del conjunto de prueba terminaría influyendo en el preprocesamiento y
en el modelo, y las métricas de evaluación serían engañosamente optimistas: eso es
data leakage.

El pipeline de scikit-learn codifica esta disciplina de forma natural:

- `fit_transform` sobre el entrenamiento: aprende los parámetros y transforma.
- `transform` sobre la prueba: aplica los parámetros ya aprendidos, sin volver a
  ajustar nada.

Además, eliminar los duplicados antes del split cierra otra vía de fuga (registros
idénticos en ambos conjuntos), y en la construcción de las variables derivadas solo se
usan predictoras del mismo registro: nunca el objetivo ni información futura. Más
adelante, en las validaciones, se comprueba que los parámetros aprendidos corresponden
efectivamente al conjunto de entrenamiento.

## 🧮 Identificación de tipos de variables

El análisis exploratorio dejó una división clara de las variables predictoras según su
distribución, y esa división define qué transformación recibe cada grupo:

- **Pruebas hepáticas sesgadas (5):** `Total_Bilirubin`, `Direct_Bilirubin`,
  `Alkaline_Phosphotase`, `Alamine_Aminotransferase` y `Aspartate_Aminotransferase`,
  con asimetrías entre 3,2 y 10,6. Reciben imputación por mediana, transformación
  logarítmica y escalado.
- **Variables numéricas simétricas (4):** `Age`, `Total_Protiens`, `Albumin` y
  `Albumin_and_Globulin_Ratio`, con asimetrías entre -0,24 y 1,03. Reciben imputación
  por mediana y escalado; no necesitan transformación logarítmica.
- **Variable categórica nominal (1):** `Gender`, con las categorías `Male` y `Female`
  sin orden entre ellas. Recibe imputación por moda y codificación one-hot.

Definimos también las columnas de origen de los cocientes clínicos que se crean en la
siguiente sección.

In [11]:
# pruebas hepáticas con fuerte asimetría derecha (según el EDA)
cols_numericas_sesgadas = [
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
]

# variables numéricas con distribución aproximadamente simétrica
cols_numericas_simetricas = [
    "Age",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
]

# variable categórica nominal (sin orden entre categorías)
cols_categoricas = ["Gender"]

# columnas de origen de los cocientes clínicos derivados
cols_cocientes = [
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
]

## 🎯 Selección de atributos

Ninguna de las 10 variables predictoras se elimina en esta etapa, y la decisión está
justificada criterio por criterio:

- **No hay identificadores** ni columnas sin contenido informativo; el EDA lo descartó
  al revisar la cardinalidad de cada variable.
- **Ninguna variable genera leakage**: ninguna se deriva del objetivo ni contiene
  información del futuro; todas son resultados de laboratorio disponibles en el momento
  de la predicción.
- **Los problemas de calidad** (duplicados, filas sin etiqueta, bilirrubinas
  imposibles) afectan a filas concretas y ya se resolvieron en la limpieza; no obligan
  a descartar columnas completas.
- **La redundancia se conserva deliberadamente.** El EDA midió correlaciones altas entre
  bilirrubina total y directa (0,89), ALT y AST (0,78), proteínas totales y albúmina
  (0,77), y albúmina y ratio A/G (0,69). Eliminar variables ahora sería una decisión
  prematura: los modelos basados en árboles toleran bien la multicolinealidad, y la
  comparación entre modelos con y sin variables redundantes corresponde a la etapa de
  modelado. De hecho, los cocientes clínicos de la siguiente sección aprovechan parte
  de esa redundancia para convertirla en información nueva.
- **`Total_Protiens`** no distingue las clases por sí sola (Mann-Whitney p = 0,65 en el
  análisis bivariable), pero se conserva porque puede aportar en combinación con las
  demás; descartarla es una decisión que se tomará con evidencia de modelado y no por
  intuición.

## 👷 Ingeniería de atributos: variables clínicas derivadas

A partir del conocimiento del dominio y de las relaciones encontradas en el EDA creamos
dos variables nuevas, ambas cocientes entre pruebas del mismo registro del paciente:

1. **`Ratio_Bilirrubina_Directa`** = bilirrubina directa / bilirrubina total. Representa
   la fracción conjugada de la bilirrubina, un indicador estándar en la interpretación
   de paneles hepáticos. Aunque las dos bilirrubinas están muy correlacionadas
   (r = 0,89), el cociente captura su relación relativa y no su magnitud, y le aporta al
   modelo una información distinta a la de las dos variables originales.
2. **`Ratio_De_Ritis`** = AST / ALT. Es el cociente de De Ritis, un marcador clínico
   conocido cuyo desplazamiento se asocia a distintos tipos de daño hepático. Igual que
   en el caso anterior, convierte la redundancia entre las dos transaminasas
   (r = 0,78) en una señal nueva.

Ambas variables se calculan únicamente con predictoras del mismo registro, disponibles
en el momento de la predicción: no emplean el objetivo, ni datos de otros pacientes, ni
información futura. Si alguna prueba base falta, el cociente queda como faltante y lo
imputa el paso siguiente del pipeline con la mediana del entrenamiento; y si en datos
nuevos apareciera una división por cero, se trataría como faltante por robustez.

Un detalle importante: como las 3 filas con bilirrubina directa mayor que la total ya se
descartaron, el cociente directa/total queda garantizado entre 0 y 1. En el conjunto de
entrenamiento, el cociente de De Ritis toma valores entre 0,09 y 8,92, un rango
clínicamente razonable.

Para que estas variables se creen dentro del flujo reproducible de scikit-learn (y no a
mano sobre el DataFrame), definimos un transformador propio que respeta la interfaz
`fit` / `transform` / `get_feature_names_out`:

In [12]:
class CocientesClinicos(BaseEstimator, TransformerMixin):
    """Transformador compatible con scikit-learn que crea variables clínicas derivadas.

    Genera, por paciente, dos cocientes a partir de las pruebas hepáticas:
    - Ratio_Bilirrubina_Directa: bilirrubina directa / bilirrubina total.
    - Ratio_De_Ritis: AST / ALT (cociente de De Ritis).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        cocientes = pd.DataFrame(
            {
                "Ratio_Bilirrubina_Directa": X["Direct_Bilirubin"] / X["Total_Bilirubin"],
                "Ratio_De_Ritis": X["Aspartate_Aminotransferase"] / X["Alamine_Aminotransferase"],
            },
            index=X.index,
        )
        # por robustez ante datos nuevos: una división por cero se trata como faltante
        return cocientes.replace([np.inf, -np.inf], np.nan)

    def get_feature_names_out(self, input_features=None):
        return np.array(["Ratio_Bilirrubina_Directa", "Ratio_De_Ritis"])

## 🔧 Transformadores y pipelines

Cada grupo de variables se procesa con su propio `Pipeline`, y después todos se
integran en un `ColumnTransformer`. Las decisiones de diseño de cada rama:

- **Imputación por mediana en las numéricas:** con colas tan marcadas, la media queda
  arrastrada por los valores extremos (por ejemplo, AST tiene media 111 frente a
  mediana 43); la mediana es un valor central robusto y más representativo del paciente
  típico.
- **Transformación `log1p` solo en las pruebas sesgadas:** comprime las colas derechas
  reduciendo la asimetría (en el entrenamiento pasa de un rango de 3,2–9,7 a uno de
  1,2–1,8) sin descartar a ningún paciente y manteniendo el orden de los valores. Se usa
  `log1p` en lugar de `log` por seguridad numérica con valores cercanos a cero.
- **`StandardScaler` en todas las numéricas:** la siguiente etapa del proyecto evaluará
  varios modelos (regresión logística, k-vecinos, SVM, árboles); los basados en
  distancias o en regularización necesitan variables en escalas comparables, y a los
  árboles el escalado no les afecta. Tras el logaritmo las distribuciones quedan
  aproximadamente simétricas, momento en el que la estandarización es la opción natural.
  No se usa `MinMaxScaler` porque los valores extremos reales de las pruebas hepáticas
  comprimirían el rango útil del resto de pacientes.
- **Imputación por moda y `OneHotEncoder` en `Gender`:** al ser una variable nominal sin
  orden entre categorías, se codifica como columnas binarias en lugar de asignar números
  arbitrarios, lo que introduciría un orden inexistente. Con
  `handle_unknown="infrequent_if_exist"` el pipeline no falla si llega una categoría
  nueva en datos de producción.

In [13]:
# pruebas hepáticas sesgadas: mediana + log1p + estandarización
pipe_sesgadas = Pipeline(
    steps=[
        ("imputacion_mediana", SimpleImputer(strategy="median")),
        ("transformacion_log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("escalado", StandardScaler()),
    ]
)

# variables numéricas simétricas: mediana + estandarización
pipe_simetricas = Pipeline(
    steps=[
        ("imputacion_mediana", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]
)

# cocientes clínicos: creación + mediana + estandarización
pipe_cocientes = Pipeline(
    steps=[
        ("crear_cocientes", CocientesClinicos()),
        ("imputacion_mediana", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]
)

# variable categórica: moda + one-hot encoding
pipe_categoricas = Pipeline(
    steps=[
        ("imputacion_moda", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False)),
    ]
)

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("hepaticas_sesgadas", pipe_sesgadas, cols_numericas_sesgadas),
        ("numericas_simetricas", pipe_simetricas, cols_numericas_simetricas),
        ("cocientes_clinicos", pipe_cocientes, cols_cocientes),
        ("categoricas", pipe_categoricas, cols_categoricas),
    ],
    verbose_feature_names_out=True,
)

# salida como DataFrame de pandas para conservar los nombres de las columnas
preprocessor.set_output(transform="pandas")
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('hepaticas_sesgadas', ...), ('numericas_simetricas', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``tr

El `ColumnTransformer` ensamblado organiza el preprocesamiento en cuatro ramas:

1. **`hepaticas_sesgadas`**: las 5 pruebas hepáticas → imputación por mediana →
   log1p → estandarización.
2. **`numericas_simetricas`**: edad y variables de proteínas → imputación por
   mediana → estandarización.
3. **`cocientes_clinicos`**: creación de los 2 cocientes clínicos a partir de las 4
   columnas base → imputación por mediana → estandarización.
4. **`categoricas`**: `Gender` → imputación por moda → one-hot.

Las 10 columnas predictoras se consumen en alguna rama y el resultado son 13
columnas: las 9 numéricas procesadas, las 2 derivadas y las 2 columnas del one-hot.
Las ramas 1 y 3 comparten columnas de origen (las bilirrubinas y las transaminasas),
algo que el `ColumnTransformer` permite sin problema porque cada rama selecciona sus
propias columnas del DataFrame original: las variables crudas entran una sola vez y
salen transformadas por su rama, además del cociente derivado.

## 🔄 Transformación de los datos

Con el preprocesador ensamblado, ajustamos **solo con el conjunto de entrenamiento** y
transformamos ambos conjuntos. Es en estas dos líneas donde se materializa la
prevención del data leakage: `fit_transform` aprende medianas, parámetros de escalado y
categorías únicamente de los 468 pacientes de entrenamiento, y `transform` aplica
exactamente esos parámetros a la prueba.

In [15]:
# el preprocesador aprende sus parámetros únicamente con los datos de entrenamiento
x_train_transformado = preprocessor.fit_transform(x_train)

# la prueba se transforma con los parámetros ya aprendidos, sin reajustar nada
x_test_transformado = preprocessor.transform(x_test)

print(f"Entrenamiento transformado: {x_train_transformado.shape}")
print(f"Prueba transformada: {x_test_transformado.shape}")

Entrenamiento transformado: (468, 13)
Prueba transformada: (117, 13)


In [16]:
nombres_columnas = preprocessor.get_feature_names_out()
print(f"Columnas resultantes: {len(nombres_columnas)}")
for nombre in nombres_columnas:
    print(" -", nombre)

Columnas resultantes: 13
 - hepaticas_sesgadas__Total_Bilirubin
 - hepaticas_sesgadas__Direct_Bilirubin
 - hepaticas_sesgadas__Alkaline_Phosphotase
 - hepaticas_sesgadas__Alamine_Aminotransferase
 - hepaticas_sesgadas__Aspartate_Aminotransferase
 - numericas_simetricas__Age
 - numericas_simetricas__Total_Protiens
 - numericas_simetricas__Albumin
 - numericas_simetricas__Albumin_and_Globulin_Ratio
 - cocientes_clinicos__Ratio_Bilirrubina_Directa
 - cocientes_clinicos__Ratio_De_Ritis
 - categoricas__Gender_Female
 - categoricas__Gender_Male


In [17]:
x_train_transformado.head()

,hepaticas_sesgadas__Total_Bilirubin,hepaticas_sesgadas__Direct_Bilirubin,hepaticas_sesgadas__Alkaline_Phosphotase,hepaticas_sesgadas__Alamine_Aminotransferase,hepaticas_sesgadas__Aspartate_Aminotransferase,numericas_simetricas__Age,numericas_simetricas__Total_Protiens,numericas_simetricas__Albumin,numericas_simetricas__Albumin_and_Globulin_Ratio,cocientes_clinicos__Ratio_Bilirrubina_Directa,cocientes_clinicos__Ratio_De_Ritis,categoricas__Gender_Female,categoricas__Gender_Male
259,3.267730,3.322052,0.246943,0.435488,0.829791,-0.193477,-1.209844,-1.303832,-1.080350,0.913807,0.528199,0.0,1.0
369,-0.617373,-0.631088,-1.185334,-0.649889,-0.762137,0.792741,0.504204,1.131302,1.163359,-0.754672,-0.420884,1.0,0.0
608,0.360684,0.381866,0.181656,1.140559,0.267549,0.114716,-0.257596,0.234147,0.522299,1.036929,-0.864729,0.0,1.0
301,-0.543983,-0.631088,0.214590,-0.792813,-0.588984,0.361271,0.218529,0.105983,-0.439290,-0.969664,-0.037714,1.0,0.0
4,0.741976,0.795561,-0.446229,-0.523182,0.061093,1.655682,0.789878,-0.919337,-1.721410,1.279484,0.711597,0.0,1.0


La matriz final de entrenamiento tiene 468 filas y 13 columnas. Los prefijos de los
nombres (`hepaticas_sesgadas__`, `numericas_simetricas__`, `cocientes_clinicos__`,
`categoricas__`) permiten rastrear de qué rama del pipeline salió cada variable. Los
primeros registros ya muestran el resultado esperado: variables numéricas centradas
alrededor de cero y columnas del género en 0/1.

## ✅ Validaciones del procesamiento

Comprobamos que el pipeline hizo lo que se esperaba: dimensiones conservadas,
imputación completa, escalado correcto, codificación one-hot bien formada, parámetros
aprendidos solo del entrenamiento, efectividad de la transformación logarítmica y
capacidad de transformar datos nuevos sin reajustar nada.

In [18]:
# dimensiones conservadas y ausencia de valores faltantes tras la imputación
assert x_train_transformado.shape[0] == x_train.shape[0]
assert x_test_transformado.shape[0] == x_test.shape[0]
assert x_train_transformado.isna().sum().sum() == 0
assert x_test_transformado.isna().sum().sum() == 0
print("Dimensiones conservadas y cero valores faltantes en entrenamiento y prueba")

Dimensiones conservadas y cero valores faltantes en entrenamiento y prueba


In [19]:
# en las columnas escaladas del entrenamiento: media ~0 y desviación ~1
columnas_escaladas = [
    nombre for nombre in x_train_transformado.columns if not nombre.startswith("categoricas__")
]
resumen_escalado = x_train_transformado[columnas_escaladas].agg(["mean", "std"]).T
resumen_escalado.round(3)

,mean,std
hepaticas_sesgadas__Total_Bilirubin,-0.0,1.001
hepaticas_sesgadas__Direct_Bilirubin,0.0,1.001
hepaticas_sesgadas__Alkaline_Phosphotase,0.0,1.001
hepaticas_sesgadas__Alamine_Aminotransferase,0.0,1.001
hepaticas_sesgadas__Aspartate_Aminotransferase,-0.0,1.001
numericas_simetricas__Age,0.0,1.001
numericas_simetricas__Total_Protiens,0.0,1.001
numericas_simetricas__Albumin,-0.0,1.001
numericas_simetricas__Albumin_and_Globulin_Ratio,-0.0,1.001
cocientes_clinicos__Ratio_Bilirrubina_Directa,0.0,1.001


In [20]:
# el one-hot está bien formado: exactamente una categoría activa por paciente
suma_onehot_train = (
    x_train_transformado["categoricas__Gender_Female"]
    + x_train_transformado["categoricas__Gender_Male"]
)
suma_onehot_test = (
    x_test_transformado["categoricas__Gender_Female"]
    + x_test_transformado["categoricas__Gender_Male"]
)
assert bool((suma_onehot_train == 1).all())
assert bool((suma_onehot_test == 1).all())
print("One-hot correcto: cada paciente tiene exactamente una categoría de género activa")
proporcion_hombres = x_train_transformado["categoricas__Gender_Male"].mean()
print(f"Proporción de hombres en entrenamiento (tras la imputación): {proporcion_hombres:.1%}")

One-hot correcto: cada paciente tiene exactamente una categoría de género activa
Proporción de hombres en entrenamiento (tras la imputación): 74.4%


In [21]:
# las medianas aprendidas por el imputador son las del entrenamiento,
# no las del dataset completo: evidencia de que no hay fuga de información
medianas_aprendidas = pd.Series(
    preprocessor.named_transformers_["numericas_simetricas"]
    .named_steps["imputacion_mediana"]
    .statistics_,
    index=cols_numericas_simetricas,
)
comparacion_medianas = pd.DataFrame(
    {
        "mediana_aprendida_pipeline": medianas_aprendidas,
        "mediana_entrenamiento": x_train[cols_numericas_simetricas].median(),
        "mediana_todos_los_datos": X_features[cols_numericas_simetricas].median(),
    }
)
assert np.allclose(
    comparacion_medianas["mediana_aprendida_pipeline"].astype("float64"),
    comparacion_medianas["mediana_entrenamiento"].astype("float64"),
)
print("El imputador aprendió exactamente las medianas del conjunto de entrenamiento")
comparacion_medianas

El imputador aprendió exactamente las medianas del conjunto de entrenamiento


,mediana_aprendida_pipeline,mediana_entrenamiento,mediana_todos_los_datos
Age,46.0,46.0,45.0
Total_Protiens,6.5,6.5,6.6
Albumin,3.1,3.1,3.1
Albumin_and_Globulin_Ratio,0.9,0.9,0.93


Las medianas que aprendió el imputador coinciden exactamente con las del conjunto de
entrenamiento (por ejemplo, 46 años para la edad). Nótese que la mediana del dataset
completo es distinta (45 años para la edad, 6,6 para las proteínas totales): que el
pipeline no aprendiera esos valores es la evidencia concreta de que el conjunto de
prueba no participó en el ajuste.

In [22]:
# la transformación logarítmica reduce la asimetría de las pruebas hepáticas
columnas_log = [f"hepaticas_sesgadas__{columna}" for columna in cols_numericas_sesgadas]
asimetria = pd.DataFrame(
    {
        "asimetria_original": x_train[cols_numericas_sesgadas].astype("float64").skew().to_numpy(),
        "asimetria_tras_log": x_train_transformado[columnas_log].skew().to_numpy(),
    },
    index=cols_numericas_sesgadas,
)
asimetria.round(2)

,asimetria_original,asimetria_tras_log
Total_Bilirubin,5.07,1.79
Direct_Bilirubin,3.23,1.79
Alkaline_Phosphotase,3.55,1.35
Alamine_Aminotransferase,6.27,1.51
Aspartate_Aminotransferase,9.66,1.22


La transformación logarítmica reduce la asimetría de las cinco pruebas hepáticas de un
rango de 3,2–9,7 a uno de 1,2–1,8. No la elimina por completo (queda una asimetría
moderada, coherente con que la mayoría de los pacientes tiene valores bajos), pero las
distribuciones dejan de estar dominadas por los extremos, que es lo que necesitan el
escalado y los modelos sensibles a la forma de las variables.

In [23]:
# el pipeline transforma un registro nuevo sin reajustarse,
# incluso con una categoría de género no vista en el entrenamiento
paciente_nuevo = pd.DataFrame(
    {
        "Age": [52],
        "Gender": ["Unknown"],
        "Total_Bilirubin": [2.1],
        "Direct_Bilirubin": [0.9],
        "Alkaline_Phosphotase": [280.0],
        "Alamine_Aminotransferase": [45.0],
        "Aspartate_Aminotransferase": [38.0],
        "Total_Protiens": [6.1],
        "Albumin": [2.9],
        "Albumin_and_Globulin_Ratio": [0.8],
    }
)

paciente_transformado = preprocessor.transform(paciente_nuevo)
print(f"El pipeline transformó el dato nuevo sin errores: {paciente_transformado.shape}")
paciente_transformado.T

El pipeline transformó el dato nuevo sin errores: (1, 13)


,0
hepaticas_sesgadas__Total_Bilirubin,0.120521
hepaticas_sesgadas__Direct_Bilirubin,0.084396
hepaticas_sesgadas__Alkaline_Phosphotase,0.214590
hepaticas_sesgadas__Alamine_Aminotransferase,0.031858
hepaticas_sesgadas__Aspartate_Aminotransferase,-0.362984
numericas_simetricas__Age,0.422909
numericas_simetricas__Total_Protiens,-0.352820
numericas_simetricas__Albumin,-0.278512
numericas_simetricas__Albumin_and_Globulin_Ratio,-0.439290
cocientes_clinicos__Ratio_Bilirrubina_Directa,0.627420


El pipeline transforma el registro nuevo sin errores y sin volver a ajustarse. Nótese la
fila de `Gender`: al ser una categoría que no apareció en el entrenamiento, ambas
columnas del one-hot quedan en 0, que es exactamente el comportamiento de
`handle_unknown="infrequent_if_exist"`; en producción esto significa que un valor
inesperado de género no rompe el procesamiento. Las demás columnas quedan en la misma
escala del entrenamiento porque se transformaron con sus mismos parámetros.

## 📊 Análisis de resultados

El preprocesamiento quedó listo y validado. Recapitulando lo que hace el flujo
construido en este notebook:

- **Limpieza estructural (antes del split):** de 663 filas se pasa a 585 tras eliminar
  60 duplicados exactos (para evitar registros idénticos en entrenamiento y prueba),
  15 filas sin etiqueta en el objetivo (que no se imputa) y 3 filas con bilirrubina
  directa mayor que la total (errores de captura fisiológicamente imposibles).
- **División estratificada:** 468 pacientes para entrenamiento y 117 para prueba,
  conservando el desbalance de clases (70,7 % / 70,9 % de pacientes con problemas de
  hígado).
- **Dentro del pipeline:** imputación por mediana de las numéricas y por moda del
  género, transformación log1p de las 5 pruebas hepáticas sesgadas, creación de 2
  variables clínicas derivadas (ratio de bilirrubina directa y cociente de De Ritis),
  estandarización de todas las numéricas y codificación one-hot de `Gender`.
- **Resultado:** matrices de 13 columnas sin valores faltantes, con las variables
  numéricas estandarizadas (media 0 y desviación 1 en el entrenamiento) y la variable
  categórica correctamente codificada.

Todas las transformaciones que aprenden parámetros se ajustaron únicamente con el
entrenamiento y se aplicaron a la prueba con `transform`, de modo que las métricas que
se obtengan en la etapa de modelado sobre el conjunto de prueba serán una estimación
honesta de la generalización. El mismo pipeline puede procesar registros nuevos de un
paciente sin repetir manualmente ningún paso, como se comprobó con el paciente de
ejemplo.

## 💡 Propuestas e ideas

- En la etapa de modelado, comparar modelos lineales (regresión logística) con árboles
  y ensembles; la comparación también servirá para evaluar si las variables redundantes
  (las dos bilirrubinas, las dos transaminasas) aportan o conviene eliminarlas.
- Evaluar estrategias de manejo del desbalance de clases (ponderación de clases,
  sobremuestreo) y reportar métricas por clase, AUC-ROC y resultados separados por
  género, dada la baja representación de mujeres en el dataset.
- Como alternativas a la imputación por mediana, probar `KNNImputer` o
  `IterativeImputer` dentro del mismo `ColumnTransformer` y comparar el impacto en el
  desempeño.
- Persistir el pipeline ajustado con `joblib` cuando se entrene el modelo final, para
  reutilizarlo en producción sin reprocesar nada manualmente.
- Si en el futuro se consigue la fuente original de las 3 filas con bilirrubinas
  inconsistentes, corregirlas en la etapa intermedia y volver a ejecutar este
  preprocesamiento.

## 📖 Referencias

- Documentación de scikit-learn sobre pipelines y transformadores compuestos:
  <https://scikit-learn.org/stable/modules/compose.html>
- Documentación de scikit-learn sobre imputación de valores faltantes:
  <https://scikit-learn.org/stable/modules/impute.html>
- Documentación de `OneHotEncoder` y el manejo de categorías desconocidas:
  <https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html>
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow (2ª edición),
  Aurélien Géron - capítulo 2, sobre pipelines de preparación de datos.
- Dataset ILPD (Indian Liver Patient Dataset):
  <https://archive.ics.uci.edu/dataset/225/ilpd+indian+liver+patient+dataset>